# Mechanistic Interpretability - Induction Circuits

We will explore the concept of induction circuits in neural networks, particularly in transformer architectures. Induction circuits are mechanisms that allow models to recognize and replicate patterns in sequences, enabling them to generalize from learned data.

### Load transformer model and tokenizer

In [ ]:
import functools
import torch
import transformers
from jaxtyping import Float, Int
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    ActivationCache,
    FactoredMatrix,
    utils,
)
from typing import List
import einops
from transformer_lens.hook_points import HookPoint
from circuitsvis.attention import attention_patterns
from huggingface_hub import hf_hub_download
import plotly.express as px

device = torch.device(
    "cpu"
    # "mps" if torch.backends.mps.is_available() else "cpu"
)
torch.set_grad_enabled(False) # Saves computation time

# TransformerLens: Introduction

> - Load and run a `HookedTransformer` model
> - Understand the basic architecture of these models
> - Use the model's tokenizer to convert text to tokens, and vice versa
> - Know how to cache activations, and to access activations from the cache
> - Use `circuitsvis` to visualise attention heads

Use gpt2_small.cfg to find the following, for your GPT-2 Small model:

- Number of layers
- Number of heads per layer
- Maximum context window

In [ ]:
# Load HookedTransformer model
gpt2_small = HookedTransformer.from_pretrained("gpt2-small", dtype=torch.float32).to(device)
gpt2_cfg = gpt2_small.cfg

In [ ]:
print(gpt2_cfg.n_layers)
print(gpt2_cfg.n_heads)
print(gpt2_cfg.n_ctx)

In [ ]:
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model the model out, let's find the loss on this paragraph!"""

loss = gpt2_small(model_description_text, return_type="loss")
print("Model loss:", loss)

In [ ]:
print(gpt2_small.to_str_tokens("gpt2"))
print(gpt2_small.to_str_tokens(["gpt2", "gpt2"]))
print(gpt2_small.to_tokens("gpt2"))
print(gpt2_small.to_string([50256,    70,   457,    17]))

### Number of correct predictions

In [ ]:
logits = gpt2_small(model_description_text, return_type="logits")
prediction = logits.argmax(dim=-1).squeeze()[:-1]

# [1:] is added to shift the tokens to left so we compare the result with next token not the current one
expected = gpt2_small.to_tokens(model_description_text).squeeze()[1:]

is_correct = prediction == expected
# Count matching values (compare only the overlapping portion)
print(f"Model accuracy: {is_correct.sum()}/{len(expected)}")
print(f"Correct tokens: {gpt2_small.to_str_tokens(prediction[is_correct])}")

### Caching all activations


In [ ]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_text, remove_batch_dim=True)

print(type(gpt2_logits), type(gpt2_cache))

### Analyzing the cache

In [ ]:
attn_patterns_shorthand = gpt2_cache["pattern", 0]
attn_patterns_full = gpt2_cache["blocks.0.attn.hook_pattern"]

torch.testing.assert_close(attn_patterns_shorthand, attn_patterns_full)

### Verify activations

In [ ]:
gpt2_cache.keys()

In [ ]:
layer0_pattern_from_cache = gpt2_cache["pattern", 0]
layer0_k = gpt2_cache["k", 0]
layer0_q = gpt2_cache["q", 0]

query_pos, n_head, d_head = layer0_q.shape

# Attn score
attn_score = einops.einsum(layer0_q, layer0_k, "query_pos n_head d_head, key_pos n_head d_head -> n_head query_pos key_pos")
attn_score = attn_score / d_head**0.5
mask = torch.tril(torch.ones((query_pos, query_pos))).bool()
attn_score = torch.where(mask, attn_score, -1e9)
attn_score = attn_score.softmax(dim=-1)

torch.testing.assert_close(attn_score, layer0_pattern_from_cache)

### Circuit visualization diagram

In [ ]:
for layer in range(gpt2_cfg.n_layers):
    attn_pattern = gpt2_cache["pattern", layer]
    display(
        attention_patterns(
            tokens=gpt2_small.to_str_tokens(gpt2_text),
            attention=attn_pattern,
        )
    )

### TODOs:

- Induction circuits explanation
    - Previous token attention head - Done
    - Current token attention head - Done
    - Induction head attention pattern - Done
    - Implementing detectors - Done
    - Induction head analysis
- Logit attribution diagram for induction heads
- Hooks using HookedTransformer
- Causal intervention/Ablation studies on induction heads 

### Interpreting two layer only model

In [ ]:
cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
)
REPO_ID = "callummcdougall/attn_only_2L_half"
FILE_NAME = "attn_only_2L_half.pth"
weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILE_NAME)
model = HookedTransformer(cfg)
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

### Visualize and inspect attention patterns

In [ ]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

In [ ]:
for layer in range(cfg.n_layers):
    attn_pattern = cache["pattern", layer]
    display(
        attention_patterns(
            tokens=model.to_str_tokens(text),
            attention=attn_pattern,
        )
    )

### Observations on attention patterns

Potentially:
- Layer 0 Head 3 - Attends to <endoftext> token that appears at the beginning of the sequence.
- Layer 0 Head 7 - Previous token attention head. Attends to the previous occurrence of the current token.
- Layer 1 Head 4 - Induction head that attends to the next token after the previous occurrence of the current token.
- Layer 1 Head 10 - Same as above.

Before we validate the observations, we should build detectors for the induction heads to confirm their functionality.

In [ ]:
def find_prev_token_attn_heads(cache) -> List[str]:
    threshold = 0.5
    result = []
    for layer in range(cfg.n_layers):
        attn_pattern = cache["pattern", layer] # n_heads, query_pos, key_pos
        for head in range(cfg.n_heads):
            score = attn_pattern[head].diag(diagonal=-1).mean()
            if score >= threshold:
                result.append(f"{layer}.{head}")
    
    return result

def find_curr_token_attn_heads(cache) -> List[str]:
    threshold = 0.3
    result = []
    for layer in range(cfg.n_layers):
        attn_pattern = cache["pattern", layer] # n_heads, query_pos, key_pos
        for head in range(cfg.n_heads):
            score = attn_pattern[head].diag(diagonal=0).mean()
            if score >= threshold:
                result.append(f"{layer}.{head}")
    
    return result

def find_first_token_attn_head(cache) -> List[str]:
    threshold = 0.5
    result = []
    for layer in range(cfg.n_layers):
        attn_pattern = cache["pattern", layer]
        for head in range(cfg.n_heads):
            score = attn_pattern[head, :, 0].mean()
            if score >= threshold:
                result.append(f"{layer}.{head}")
    
    return result

prev_token_attn_heads = find_prev_token_attn_heads(cache)
curr_token_attn_heads = find_curr_token_attn_heads(cache)
first_token_attn_heads = find_first_token_attn_head(cache)

print(prev_token_attn_heads)
print(curr_token_attn_heads)
print(first_token_attn_heads)

### Induction head analysis

The theory on induction heads can be found [here](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html#definition-of-induction-heads)

In order to prove that we have induction heads and induction circuits present, we must show that -
1. For any sequence with repeated tokens, the attention patterns highlights attention on previous occuring copies.
2. The specific occurance of tokens in the sequence doesn't break the pattern.
3. Token loss decreases for copy of tokens which come later in the sequence. 

In [ ]:
torch.manual_seed(0)

# Ensure model is on the correct device
model = model.to(device)

def generate_random_tokens(batch, seq_len) -> torch.Tensor:
    return torch.randint(low=0, high=model.tokenizer.vocab_size, size=(batch, seq_len), device=device)

def generate_repeated_tokens(batch, seq_len) -> torch.Tensor:
    rand_tokens = generate_random_tokens(batch, seq_len)
    prefix = (torch.ones((batch, 1), device=device) * model.tokenizer.bos_token_id).long()

    return torch.concat([prefix, rand_tokens, rand_tokens], dim=-1)

def run_model_with_repeated_tokens(batch, seq_len) -> ActivationCache:
    tokens = generate_repeated_tokens(batch, seq_len)
    logits, cache = model.run_with_cache(tokens)

    return tokens, logits, cache

def get_log_probs(logits, tokens) -> torch.Tensor:
    """
    Extract log probabilities of correct next tokens at each position.
    Uses gather to index logprobs[:, :-1] along vocabulary dimension (dim=-1)
    with tokens[:, 1:], effectively computing logprobs[b, s, tokens[b, s+1]]
    """
    logprobs = logits.log_softmax(dim=-1)
    # Ensure tokens are on the same device as logits
    tokens = tokens.to(logprobs.device)
    correct_logprobs = logprobs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)
    return correct_logprobs
    

seq_len = 50
batch_size = 1
rep_tokens, rep_logits, rep_cache = run_model_with_repeated_tokens(batch_size, seq_len)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)
model.reset_hooks()
log_probs = get_log_probs(rep_logits, rep_tokens)

### Plot log probs vs Sequence length

In [ ]:
import pandas as pd

log_probs_data = log_probs.cpu().numpy()[0]
df = pd.DataFrame({
    'seq_len': range(len(log_probs_data)),
    'log_probs': log_probs_data
})

fig = px.line(df, x="seq_len", y="log_probs", title="Log probs vs Sequence length")
fig.show()

### Observation

- Log probability improves after sequence length >= 50. This shows that induction head is copying tokens from sequence < 50.

### Finding induction heads

In [ ]:
for layer in range(model.cfg.n_layers):
    attn_pattern = rep_cache["pattern", layer]
    display(attention_patterns(tokens=rep_str, attention=attn_pattern))

### Observations on attention patterns

- Layer 0 Head 7 consistently attends to the previous occurrence of the current token, confirming its role as a previous token attention head.
- Layer 1 Head 4 and Layer 1 Head 10 both exhibit attention patterns characteristic of induction heads, attending to the next token following the previous occurrence of the current token.
- The pattern of induction heads shows a shifted diagonal exactly by the length of the initial sequence segment (50 tokens in this case).
- These observations align with the theoretical understanding of induction heads and their function within transformer architectures.

### Induction head detector

In [ ]:
def find_induction_heads(cache) -> List[str]:
    threshold = 0.5
    heads = []
    for layer in range(model.cfg.n_layers):
        attn_pattern = cache["pattern", layer]
        seq_len = (attn_pattern.shape[-1] - 1)//2
        for head in range(model.cfg.n_heads):
            # A shift of -1 is needed because 
            # induction head attends to the token
            # just to right side of the previous copy 
            # of current token (which exists at an offset of seq_len)
            score = attn_pattern[head].diag(diagonal=-(seq_len-1)).mean()
            if score >= threshold:
                heads.append(f"{layer}.{head}")
    
    return heads

find_induction_heads(rep_cache)

['1.4', '1.10']